In [ ]:
import json

with open(
    "../data/candidate_profile.json",
    "r",
    encoding="utf-8"
) as f:

    candidate = json.load(f)

In [ ]:
candidate.keys()

In [ ]:
target_role = "Data Scientist"

print("Target role:", target_role)

In [ ]:
ROLE_PROFILES = {
    "data scientist": {
        "technical_skills": [
            "Python",
            "SQL",
            "Pandas",
            "NumPy",
            "Scikit-learn"
        ],

        "core_concepts": [
            "Statistics",
            "Probability",
            "Machine Learning",
            "Feature Engineering",
            "Model Evaluation",
            "Data Analysis"
        ],

        "advanced_concepts": [
            "Ensemble Methods",
            "Gradient Boosting",
            "XGBoost",
            "Cross Validation",
            "Hyperparameter Tuning",
            "A/B Testing"
        ],

        "problem_solving": [
            "Problem formulation",
            "Model selection",
            "Error analysis",
            "Trade-off reasoning"
        ],

        "communication": [
            "Explaining technical decisions",
            "Interpreting results",
            "Business reasoning"
        ]
    }
}

In [ ]:
role = ROLE_PROFILES["data scientist"]

role

In [ ]:
candidate_skills = {
    skill.lower()
    for skill in candidate["skills"]
}

role_skills = {
    skill.lower()
    for skill in role["technical_skills"]
}

matched_skills = candidate_skills.intersection(
    role_skills
)

missing_skills = role_skills - candidate_skills

In [ ]:
print("MATCHED:")
print(matched_skills)

print("\nMISSING:")
print(missing_skills)

In [ ]:
competencies = []

for category, items in role.items():

    for item in items:

        competencies.append({
            "name": item,
            "category": category
        })

In [ ]:
competencies[:10]

In [ ]:
candidate_context = {
    "skills": candidate["skills"],
    "projects": candidate["projects"],
    "experience": candidate["experience"]
}

In [ ]:
matched_count = len(matched_skills)
total_role_skills = len(role_skills)

role_match_score = (
    matched_count / total_role_skills
) * 100

print(
    f"Technical skill match: "
    f"{role_match_score:.1f}%"
)

In [ ]:
interview_blueprint = {
    "target_role": target_role,

    "competencies": competencies,

    "candidate_strengths": list(matched_skills),

    "candidate_gaps": list(missing_skills),

    "priority_topics": list(missing_skills),

    "difficulty": "medium"
}

In [ ]:
interview_blueprint

In [ ]:
print("Role:", target_role)
print("Matched skills:", matched_skills)
print("Missing skills:", missing_skills)
print("Match score:", role_match_score)
print("Competencies:", len(competencies))

# LLM role analysis

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class CompetencyAssessment(BaseModel):
    competency: str
    category: str
    status: str
    evidence: str = ""
    priority: str
    
class RoleAnalysis(BaseModel):
    target_role: str
    competencies: list[CompetencyAssessment] = Field(
        default_factory=list
    )
    strengths: list[str] = Field(default_factory=list)
    unverified_areas: list[str] = Field(default_factory=list)
    priority_topics: list[str] = Field(default_factory=list)
    recommended_difficulty: str = "medium"

In [ ]:
ROLE_ANALYSIS_PROMPT = """
You are an AI interview planning system.

Your task is to analyze a candidate against a target job role.

You are given:

1. The target role
2. A baseline competency map for that role
3. The candidate's resume-derived profile

Your job is to determine:

- Which competencies are clearly demonstrated
- Which are partially demonstrated
- Which are unverified
- What evidence exists in the candidate profile
- Which topics should receive interview priority
- What difficulty level is appropriate

Rules:

1. Never claim that a candidate is weak merely because a skill is absent from the resume.
2. Use "UNVERIFIED" when the resume provides insufficient evidence.
3. Only use evidence explicitly present in the candidate profile.
4. Do not invent candidate experience.
5. Prioritize competencies that are important for the role but insufficiently demonstrated.
6. Consider the candidate's projects and experience, not only their skills list.
7. The goal is to design an interview, not to make a final hiring decision.
8. Return only structured JSON matching the provided schema.
"""

In [ ]:
import json

role_analysis_input = {
    "target_role": target_role,
    "role_profile": role,
    "candidate_profile": candidate
}

In [ ]:
role_analysis_input_text = json.dumps(
    role_analysis_input,
    indent=2,
    ensure_ascii=False
)

print(role_analysis_input_text)

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")

groq_api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=groq_api_key)

In [ ]:
schema = RoleAnalysis.model_json_schema()

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": ROLE_ANALYSIS_PROMPT
        },
        {
            "role": "user",
            "content": f"""
Target role and candidate information:

{role_analysis_input_text}

Required JSON schema:

{json.dumps(schema, indent=2)}
"""
        }
    ],
    temperature=0
)

In [ ]:
response_text = response.choices[0].message.content

print(response_text)

In [ ]:
parsed_analysis = json.loads(response_text)

In [ ]:
role_analysis = RoleAnalysis.model_validate(
    parsed_analysis
)

In [ ]:
role_analysis

In [ ]:
for competency in role_analysis.competencies:

    print(
        competency.competency,
        "|",
        competency.status,
        "|",
        competency.priority
    )

In [ ]:
print("STRENGTHS")

for item in role_analysis.strengths:
    print("-", item)
    
print("UNVERIFIED")

for item in role_analysis.unverified_areas:
    print("-", item)
    
print("PRIORITY TOPICS")

for item in role_analysis.priority_topics:
    print("-", item)

In [ ]:
print(
    "Recommended difficulty:",
    role_analysis.recommended_difficulty
)

In [ ]:
interview_blueprint = {
    "target_role": role_analysis.target_role,
    "competencies": [
        item.model_dump()
        for item in role_analysis.competencies
    ],
    "strengths": role_analysis.strengths,
    "unverified_areas": role_analysis.unverified_areas,
    "priority_topics": role_analysis.priority_topics,
    "recommended_difficulty": role_analysis.recommended_difficulty
}

In [ ]:
with open(
    "../data/interview_blueprint.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        interview_blueprint,
        f,
        indent=2,
        ensure_ascii=False
    )